# Welcome to `SpinGlassPEPS.jl` interactive demo!

`SpinGlassPEPS.jl` is an is an collection of open-source Julia packages for numerical computation in quantum information theory. It focuses on simulating quantum annealing via PEPS tensor network. Bundled together are:
* `SpinGlassTensors.jl` - contains functions used in tensor network contractions
* `SpinGlassNetworks.jl` - creates factor graph and Ising spin-glass model
* `SpinGlassEngine.jl` - search for low energy spectrum using PEPS tensor network

In this interactive demo, we will work through a series of examples that illustrate the capabilities of SpinGlassPEPS and demonstrate the power of the used algorithm. By the end of the demo, you will have a solid understanding of how to use SpinGlassPEPS to simulate quantum spin systems on quasi-2d lattices.

## Introduction

Quantum computing has brought about a paradigm shift in the field of computer science, promising to revolutionize the way we solve complex problems that classical computers struggle with. Quantum annealing is one of the most promising approaches to quantum computing, particularly for optimization problems. The goal of quantum annealing is to minimize the energy of a given system, which can be represented as a Hamiltonian of the Ising model:

$$
H(s) = \sum_{<i,j> \in \mathcal{E}} J_{ij} s_i s_j + \sum_i h_i s_i.
$$

where $s$ is a configuration of $N$ classical spins taking values $s_i = \pm 1$
and $J_{ij}, h_i \in \mathbb{R}$ are input parameters of a given problem instance. 
Nonzero couplings $J_{ij}$ form a graph $\mathcal{E}$. Edges of $\mathcal{E}$ form a quasi-two-dimensional structure. In this package we focus in particular on the [Chimera](https://docs.dwavesys.com/docs/latest/c_gs_4.html#chimera-graph) graph with up to 2048 spins.  

Chimera graph with $126$ spins can be found bellow:

<style>
td, th {
   border: none!important;
}
</style>

| ![Simple Chimera Graph](pictures/C4.png) |
|:--:|
| *$4 \times 4$ Chimera Graph* |

## How to work with `SpinGlassPEPS.jl`.

<style>
td, th {
   border: none!important;
}
</style>

| ![workflow](pictures/workflow.png) |
|:--:|
|Workflow of `SpinGlassPEPS.jl`. |

Obviously, first step is to import `SpinGlassPEPS` package.

In [1]:
using SpinGlassPEPS

Now, we can start working with it! First we have to define instance whose low energy spectrum we wish to find. In this simple example we will start with spin chain with $3$ sites and local magnetic fields. 

In [2]:
instance = Dict((1, 1) => 0.5, (2, 2) => 0.75, (3, 3) => -0.25, 
(1, 2) => -1.0, (2, 3) => 1.0)

# nest step is to create IsingGraph object, which stores our
# instance as LabelledGraph!
ig = SpinGlassNetworks.ising_graph(instance)

# Lets see local magnetic fields and ranks of sites in the created instance 
SpinGlassNetworks.props.(Ref(ig), SpinGlassNetworks.vertices(ig))

3-element Vector{Dict{Symbol, Any}}:
 Dict(:h => 0.5, :rank => 2)
 Dict(:h => 0.75, :rank => 2)
 Dict(:h => -0.25, :rank => 2)

In next step we will create factor graph to represent our instance. `SpinGlassPEPS.jl` is designed to work with two-dimensional lattice-like structures, so we have to define number of columns - $m$, number off rows - $n$ and chimera parameter $t$.

In [3]:
m, n, t = 3, 1, 1

fg = SpinGlassNetworks.factor_graph(
    ig,
    cluster_assignment_rule = super_square_lattice((m,n,t))
)

LabelledGraphs.LabelledGraph{MetaGraphs.MetaDiGraph{Int64, Float64}, Tuple{Int64, Int64}}([(1, 1), (2, 1), (3, 1)], {3, 2} directed Int64 metagraph with Float64 weights defined by :weight (default weight 1.0), Dict((3, 1) => 3, (1, 1) => 1, (2, 1) => 2))

From factor graph we can build PEPS tensor network.To do this we must define few model parameters. The `transform` parameter allows us to rotate and reflect structure of our instance. The `β` is inverse temerature used in gibbs distribution. `bond_dim` describes the dimension of the bond index connecting one tensor in the network to another.

In [4]:
transform = SpinGlassEngine.rotation(0)
β = 1.0
bond_dim = 5


peps = SpinGlassEngine.PEPSNetwork(
    m, 
    n, 
    fg, 
    transform, 
    β = β, 
    bond_dim = bond_dim
    )

PEPSNetwork(LabelledGraphs.LabelledGraph{MetaGraphs.MetaDiGraph{Int64, Float64}, Tuple{Int64, Int64}}([(1, 1), (2, 1), (3, 1)], {3, 2} directed Int64 metagraph with Float64 weights defined by :weight (default weight 1.0), Dict((3, 1) => 3, (1, 1) => 1, (2, 1) => 2)), LabelledGraphs.LabelledGraph{LightGraphs.SimpleGraphs.SimpleGraph{Int64}, Tuple{Int64, Int64}}([(1, 1), (2, 1), (3, 1)], {3, 2} undirected simple Int64 graph, Dict((3, 1) => 3, (1, 1) => 1, (2, 1) => 2)), SpinGlassEngine.var"#10#19"(Core.Box(SpinGlassEngine.var"#2#11"())), 3, 1, 3, 1, 1.0, 32, 1.0e-8, 4, Dict{Tuple{Rational, Union{Rational{Int64}, Int64}}, Vector{Float64}}((8//3, 1) => [1.0, 1.0], (7//3, 1) => [1.0, 1.0], (5//3, 1) => [1.0, 1.0], (17//6, 1) => [1.0, 1.0], (13//6, 1) => [1.0, 1.0], (7//6, 1) => [1.0, 1.0], (4//3, 1) => [1.0, 1.0], (11//6, 1) => [1.0, 1.0]), Dict{Tuple{Union{Rational{Int64}, Int64}, Union{Rational{Int64}, Int64}}, Symbol}((3, 1) => :site, (2, 3//2) => :central_h, (13//6, 1) => :gauge_h, (11/

Now we are ready to compute low energy spectrum of our instace. We can specify how many low energy state we wish to find in `num_states` parameter.

In [28]:
num_states = 3
sol = low_energy_spectrum(peps, num_states)

#To se our solution, we have to decode factor graphs states to ising states
states = sort.(SpinGlassNetworks.decode_factor_graph_state.(Ref(fg), sol.states))
for (state, energy) ∈ zip(states, sol.energies)
    println("state: ", [s for s in values(state)], " enegy: ", energy)
end

state [-1, -1, 1] enegy: -3.5
state [-1, -1, -1] enegy: -1.0
state [1, 1, -1] enegy: -0.5


## Chimera instance

Now we will show how to work with bigger chimera instances. Intance can be specified as a `csv` or `txt` file. The rest is identical to previous examples!

In [31]:
instance = "$(@__DIR__)/instances/C4.txt"
m, n, t = 4, 4, 8
ig = SpinGlassNetworks.ising_graph(instance)

fg = SpinGlassNetworks.factor_graph(
    ig,
    cluster_assignment_rule = super_square_lattice((m,n,t))
)

transform = SpinGlassEngine.rotation(0)
β = 1.0
bond_dim = 32

peps = SpinGlassEngine.PEPSNetwork(
    m, 
    n, 
    fg, 
    transform, 
    β = β, 
    bond_dim = bond_dim
)

num_states = 10
sol = low_energy_spectrum(peps, num_states)

# Lets see lowest energy found and compare it to known groundstate
println("Energy found by SpinGlassPEPS: ", sol.energies[begin], " ground state: -210.933333")

┌ Info: Compressing state down to
│   Dcut = 32
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:57
┌ Info: Convergence
│   diff = Inf
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:64
┌ Info: Convergence
│   diff = 8.881784197001252e-16
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:64
┌ Info: Finished in 2 sweeps of 4.
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:67
Preprocesing:  75%|████████████████████████████         |  ETA: 0:00:00

┌ Info: Compressing state down to
│   Dcut = 32
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:57
┌ Info: Convergence
│   diff = Inf
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:64
┌ Info: Convergence
│   diff = 8.881784197001252e-16
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:64
┌ Info: Finished in 2 sweeps of 4.
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:67
Preprocesing: 100%|█████████████████████████████████████| Time: 0:00:01


Search:  62%|███████████████████████████                |  ETA: 0:00:00

Search: 100%|███████████████████████████████████████████| Time: 0:00:00

-210.93333400000003


In [ ]:
# We can turn off additional info with
disable_logging(LogLevel(1))

We can also work with full Dwave graph. That means $2048$ spins!

In [32]:
instance = "$(@__DIR__)/instances/C16.txt"
m, n, t = 16, 16, 8
ig = SpinGlassNetworks.ising_graph(instance)

fg = SpinGlassNetworks.factor_graph(
    ig,
    cluster_assignment_rule = super_square_lattice((m,n,t))
)

transform = SpinGlassEngine.rotation(0)
β = 1.0
bond_dim = 32

peps = SpinGlassEngine.PEPSNetwork(
    m, 
    n, 
    fg, 
    transform, 
    β = β, 
    bond_dim = bond_dim
)

num_states = 10
sol = low_energy_spectrum(peps, num_states)

# we can compare it with 
println(sol.energies[begin])

┌ Info: Compressing state down to
│   Dcut = 32
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:57


┌ Info: Convergence
│   diff = Inf
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:64


┌ Info: Convergence
│   diff = 1.0347278589506459e-13
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:64
┌ Info: Finished in 2 sweeps of 4.
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:67
Preprocesing:  19%|███████                              |  ETA: 0:00:28

┌ Info: Compressing state down to
│   Dcut = 32
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:57
┌ Info: Convergence
│   diff = Inf
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:64


┌ Info: Convergence
│   diff = 1.2434497875801753e-14
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:64
┌ Info: Finished in 2 sweeps of 4.
└ @ SpinGlassTensors C:\Users\walle\.julia\dev\SpinGlassTensors\src\compressions.jl:67
Preprocesing:  25%|██████████                           |  ETA: 0:01:26

...

Search:  91%|███████████████████████████████████████    |  ETA: 0:00:01

Search:  93%|████████████████████████████████████████   |  ETA: 0:00:00

Search:  96%|██████████████████████████████████████████ |  ETA: 0:00:00

Search: 100%|███████████████████████████████████████████| Time: 0:00:06


-3325.386687000001
